# 🧪 Week 5: Evaluating ML Models
### SheffDataSoc — DataDive

---

Welcome to the Week 5 hands-on notebook! This session covers how to **evaluate machine learning models** properly — beyond just glancing at training accuracy.

**How this notebook works:**
- 📖 Read through the explanations and code that's already written for you
- ✏️ Fill in the **blanks** marked with `###` — these are the evaluation parts you need to complete
- 🔗 Follow the documentation links when you want to go deeper

**Resources:**
- 📁 Session resources: https://github.com/sheffdatasoc/datadive-resources
- 🔗 Training guide: https://tinyurl.com/uostrain

---

## 0. Setup

Run this cell first — it installs and imports everything you'll need.

In [1]:
# All imports — run this cell first!
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Setup complete!")

✅ Setup complete!


---

## 1. Why Do We Evaluate Models?

Training a model doesn't mean it will work well in the real world. Evaluation helps us answer three key questions:

1. **Is it accurate?** Does it make correct predictions?
2. **Does it achieve the goal we care about?** Accuracy alone can be misleading.
3. **Is it safe and fair to use?** A model must work equitably across groups.

> 💡 *A spam classifier trained on poor data might let 80% of spam through in a real scenario — even if training looked great.*

---

## 2. The Dataset — A Cancer Detection Scenario

We'll simulate a **cancer detection** task — a great example of why choosing the right metric matters.

- **Class 0** = No cancer (the vast majority of patients)
- **Class 1** = Cancer detected (rare — only ~1% of patients)

This is a **class imbalanced** dataset. The model you train below is already written for you.

In [2]:
# --- Data Generation (already done for you) ---
# We create a synthetic dataset with severe class imbalance
# weights=[0.9, 0.1] means only 10% of samples are cancer-positive

X, y = make_classification(
    n_samples=10_000,
    n_features=10,
    weights=[0.9, 0.1],   # 99% negative, 1% positive
    flip_y=0.01,
    random_state=RANDOM_STATE
)

print(f"Dataset size : {len(y):,} samples")
print(f"Cancer cases : {y.sum():,} ({y.mean()*100:.1f}% of total)")
print(f"Healthy cases: {(1-y).sum():,} ({(1-y).mean()*100:.1f}% of total)")

Dataset size : 10,000 samples
Cancer cases : 1,043 (10.4% of total)
Healthy cases: 8,957 (89.6% of total)


---

## 3. Splitting Data into Train and Test Sets

One of the most fundamental steps in model evaluation is the **train/test split**.

- We **train** the model on one portion of data
- We **test** it on data the model has *never seen before*

This checks whether the model has actually *learned general patterns* — not just memorised the training examples.

The standard split ratio is **80% train / 20% test**.

> 📚 Docs: [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

### ✏️ Your Turn — Fill in the blanks below

In [3]:
# Split the data into training and test sets
# Use an 80/20 split and set random_state=RANDOM_STATE for reproducibility

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=###,           # ← What fraction should be the test set?
    random_state=RANDOM_STATE
)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")

SyntaxError: invalid syntax (3698164917.py, line 7)

---

## 4. Training the Models

We're training **two** models:

1. **A "dumb" baseline** — a model that *always* predicts "no cancer"
2. **A Logistic Regression classifier** — a real model

The baseline is important: it shows what happens when we optimise for the wrong thing.

In [ ]:
# --- Model 1: The "Always Negative" Baseline (already done for you) ---
# This model always predicts class 0 (no cancer), regardless of input
y_pred_baseline = np.zeros(len(y_test), dtype=int)

# --- Model 2: Logistic Regression (already done for you) ---
lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print("✅ Both models trained and predictions generated.")

---

## 5. Alignment with Objective — Why Accuracy Alone Lies

> *"The single most important question is: are we measuring the thing that actually matters?"*

A model that always predicts "no cancer" will be **99% accurate** on our dataset — because 99% of people don't have cancer. But it will **miss every single cancer case**.

Let's prove it.

> 📚 Docs: [`accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)

### ✏️ Your Turn

In [ ]:
# Calculate accuracy for both models
# accuracy_score(y_true, y_pred) compares the true labels against predicted labels

acc_baseline = accuracy_score(###, ###)   # ← Fill in true labels, then baseline predictions
acc_lr       = accuracy_score(###, ###)   # ← Fill in true labels, then LR predictions

print(f"Baseline accuracy  : {acc_baseline:.2%}")
print(f"Logistic Regression: {acc_lr:.2%}")
print()
print("🤔 Which model looks better based on accuracy alone?")

---

## 6. Better Metrics — Precision and Recall

The baseline model actually looks quite strong! But its accuracy has slightly misled us, we need metrics that are **aligned with the real objective**.

| Metric | What it asks | Formula |
|--------|-------------|----------|
| **Accuracy** | Of all predictions, how many were correct? | (TP + TN) / All |
| **Precision** | Of the cases we flagged as positive, how many truly were? | TP / (TP + FP) |
| **Recall** | Of all real positive cases, how many did we catch? | TP / (TP + FN) |
| **F1 Score** | Harmonic mean of Precision and Recall | 2 × (P × R) / (P + R) |

For cancer detection, **Recall is critical** — a false negative (missed cancer) is far more dangerous than a false positive.

> 📚 Docs: [`precision_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html) | [`recall_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html) | [`f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)

> ⚠️ For imbalanced datasets, always set `zero_division=0` to suppress divide-by-zero warnings.

### ✏️ Your Turn

In [ ]:
# Calculate Precision, Recall, and F1 for both models
# Each function takes (y_true, y_pred) as arguments

# --- Baseline model ---
prec_baseline   = precision_score(###, ###, zero_division=0)   # ← true labels, baseline preds
recall_baseline = recall_score(###, ###, zero_division=0)      # ← true labels, baseline preds
f1_baseline     = f1_score(###, ###, zero_division=0)          # ← true labels, baseline preds

# --- Logistic Regression model ---
prec_lr   = precision_score(###, ###, zero_division=0)         # ← true labels, LR preds
recall_lr = recall_score(###, ###, zero_division=0)            # ← true labels, LR preds
f1_lr     = f1_score(###, ###, zero_division=0)                # ← true labels, LR preds

# Pretty summary table
results = pd.DataFrame({
    'Model'    : ['Always-Negative Baseline', 'Logistic Regression'],
    'Accuracy' : [acc_baseline, acc_lr],
    'Precision': [prec_baseline, prec_lr],
    'Recall'   : [recall_baseline, recall_lr],
    'F1 Score' : [f1_baseline, f1_lr]
}).set_index('Model')

results.style.format('{:.2%}').background_gradient(cmap='RdYlGn', axis=0)

---

## 7. The Confusion Matrix

A **confusion matrix** breaks down predictions into four categories:

|  | Predicted Negative | Predicted Positive |
|---|---|---|
| **Actually Negative** | True Negative (TN) ✅ | False Positive (FP) ⚠️ |
| **Actually Positive** | False Negative (FN) ❌ | True Positive (TP) ✅ |

In medical contexts, **False Negatives (FN)** are the most dangerous — we told a patient they were healthy when they weren't.

> 📚 Docs: [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) | [`ConfusionMatrixDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html)

### ✏️ Your Turn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Confusion matrix for the Baseline model ---
cm_baseline = confusion_matrix(###, ###)                            # ← true labels, baseline preds
ConfusionMatrixDisplay(cm_baseline, display_labels=['Healthy', 'Cancer']).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Always-Negative Baseline', fontsize=13, fontweight='bold')

# --- Confusion matrix for Logistic Regression ---
cm_lr = confusion_matrix(###, ###)                                  # ← true labels, LR preds
ConfusionMatrixDisplay(cm_lr, display_labels=['Healthy', 'Cancer']).plot(
    ax=axes[1], colorbar=False, cmap='Blues'
)
axes[1].set_title('Logistic Regression', fontsize=13, fontweight='bold')

plt.suptitle('Confusion Matrices — Cancer Detection', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("🔍 Look at the bottom-left cell (False Negatives) — which model misses fewer cancer cases?")

---

## 8. Overfitting — When a Model Learns Too Much

A model might perform brilliantly on training data but fail on new data. This is called **overfitting**.

The model has memorised the training examples rather than learned general rules. To detect it:
- Evaluate on **both** the training set and the test set
- A large gap between train and test performance = overfitting

We'll use a **Decision Tree** — a model that is very prone to overfitting when grown too deep.

> 📚 Docs: [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)

### ✏️ Your Turn

In [ ]:
# We train two Decision Trees:
#   - An overfit tree (no depth limit — grows as deep as it wants)
#   - A constrained tree (max_depth=3 — kept simple)

# Both trees are already trained for you:
tree_overfit    = DecisionTreeClassifier(random_state=RANDOM_STATE)          # No limit
tree_constrained = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)  # Depth-limited

tree_overfit.fit(X_train, y_train)
tree_constrained.fit(X_train, y_train)

# --- Now evaluate each model on BOTH train and test sets ---
# Fill in the correct dataset (X_train or X_test) for each .predict() call

train_score_overfit     = accuracy_score(y_train, tree_overfit.predict(###))      # ← which set?
test_score_overfit      = accuracy_score(y_test,  tree_overfit.predict(###))      # ← which set?

train_score_constrained = accuracy_score(y_train, tree_constrained.predict(###))  # ← which set?
test_score_constrained  = accuracy_score(y_test,  tree_constrained.predict(###))  # ← which set?

overfit_df = pd.DataFrame({
    'Model'       : ['Overfit Tree (no limit)', 'Constrained Tree (depth=3)'],
    'Train Acc'   : [train_score_overfit, train_score_constrained],
    'Test Acc'    : [test_score_overfit, test_score_constrained],
    'Gap'         : [
        train_score_overfit     - test_score_overfit,
        train_score_constrained - test_score_constrained
    ]
}).set_index('Model')

overfit_df.style.format('{:.2%}').background_gradient(subset=['Gap'], cmap='RdYlGn_r')

In [ ]:
# Visualise the overfitting gap
labels  = ['Overfit Tree', 'Constrained Tree']
train_s = [train_score_overfit, train_score_constrained]
test_s  = [test_score_overfit, test_score_constrained]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, train_s, width, label='Train Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, test_s,  width, label='Test Accuracy',  color='coral')

ax.set_ylabel('Accuracy')
ax.set_title('Overfitting: Train vs Test Accuracy', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0.85, 1.01)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))

plt.tight_layout()
plt.show()
print("📌 A large gap between the blue and red bars = overfitting.")

---

## 9. Full Classification Report

The `classification_report` function gives you a **complete summary** of all key metrics broken down by class. It's a great first step in any evaluation.

> 📚 Docs: [`classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)

### ✏️ Your Turn

In [ ]:
# Print a full classification report for the Logistic Regression model
# Hint: classification_report(y_true, y_pred, target_names=[...])

print("=== Logistic Regression — Full Classification Report ===")
print(
    classification_report(
        ###,                                    # ← true labels
        ###,                                    # ← LR predictions
        target_names=['Healthy', 'Cancer']
    )
)
print()
print("🔎 Pay attention to the Cancer row — Recall is our most important metric here.")

---

## 10. Bias — Checking Fairness Across Groups

A model might perform well **overall** but unfairly disadvantage certain groups. This is what happened with [Amazon's hiring algorithm](https://www.reuters.com/article/us-amazon-com-jobs-automation-insight/amazon-scraps-secret-ai-recruiting-tool-that-showed-bias-against-women-idUSKCN1MK08G) — it penalised CVs that included the word *"women's"* (e.g. women's chess club).

Best practice: **always check metrics broken down by subgroups** (e.g. age, gender, ethnicity).

> 📚 Docs: [`groupby` + `apply`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)

### ✏️ Your Turn

In [ ]:
# We simulate two demographic groups (already done for you)
np.random.seed(99)
groups = np.random.choice(['Group A', 'Group B'], size=len(y_test), p=[0.6, 0.4])

bias_df = pd.DataFrame({
    'group'    : groups,
    'y_true'   : y_test,
    'y_pred_lr': y_pred_lr
})

# Calculate recall for each group separately
# We want to check: does the model catch cancer equally well across groups?

def group_recall(df):
    return recall_score(df['y_true'], df['###'], zero_division=0)  # ← which column holds LR predictions?

group_metrics = bias_df.groupby('###').apply(group_recall)         # ← which column defines the groups?

print("=== Recall by Demographic Group ===")
for group, rec in group_metrics.items():
    print(f"  {group}: {rec:.2%} recall")

print()
print("⚠️  A large gap in recall between groups would mean the model is missing more cancer")
print("   cases in one group than another — a serious ethical problem.")

---

## 11. Building Ethical Models — Questions to Always Ask

Good evaluation isn't just about numbers. Before deploying any model, work through this checklist:

| Question | Why it matters |
|----------|----------------|
| ✅ Does the metric match the real goal? | Optimising the wrong thing gives a useless model |
| ✅ Does performance hold on unseen data? | Train/test split catches overfitting |
| ✅ Are certain groups affected differently? | Fairness is not optional |
| ✅ What are the consequences of errors? | False negatives vs false positives have different costs |
| ✅ Can we explain the model's decisions? | Explainability builds trust and enables accountability |

---

## 12. The Evaluation Workflow

Here is the complete workflow to follow every time you evaluate a model:

```
1. Train Model
       ↓
2. Test on Unseen Data  ← Never skip this!
       ↓
3. Calculate Metrics aligned with the real objective
       ↓
4. Check Bias across subgroups
       ↓
5. Review Ethical Risks
       ↓
6. Decide: Safe to Deploy?
       ↓
   Yes → 🚀 Deploy      No → 🔁 Retrain
```

If the model fails, revisit your:
- **Dataset** — more data, better quality, fix imbalance
- **Objective / Metric** — are you optimising for the right thing?
- **Model Type** — try a different algorithm

---

## 13. Summary & Key Takeaways

Run the cell below to generate your personal summary report for this session.

In [ ]:
print("=" * 55)
print("   WEEK 5 — SESSION SUMMARY")
print("=" * 55)

print(f"""
📊 Dataset
   {len(y):,} samples | {y.mean()*100:.1f}% positive (cancer)

🎯 Alignment with Objective
   Baseline accuracy  : {acc_baseline:.2%}  ← looks great, but...
   Baseline recall    : {recall_baseline:.2%}  ← catches ZERO cancer cases!
   LR recall          : {recall_lr:.2%}  ← much better for the real goal

🔁 Overfitting
   Overfit tree gap   : {train_score_overfit - test_score_overfit:.2%} (train - test)
   Constrained gap    : {train_score_constrained - test_score_constrained:.2%} (train - test)

⚖️  Fairness
   Group A recall     : {group_metrics.get('Group A', 0):.2%}
   Group B recall     : {group_metrics.get('Group B', 0):.2%}
""")

print("=" * 55)
print("Key Takeaways")
print("=" * 55)
print("""
  1. A good model is aligned with the real objective —
     not just maximising accuracy.

  2. Always evaluate on unseen (test) data to detect
     overfitting.

  3. Check performance across subgroups to ensure
     fairness.

  4. Consider the cost of different errors — FP vs FN
     have very different consequences.

  5. If the model fails, revisit your dataset,
     objective, or model type.
""")
print("=" * 55)
print("\n🔗 https://github.com/sheffdatasoc/datadive-resources")

---

## 🎉 You're Done!

Great work completing the Week 5 notebook. Here's what you practised:

- ✅ Splitting data into train/test sets
- ✅ Calculating accuracy, precision, recall, and F1
- ✅ Interpreting confusion matrices
- ✅ Detecting overfitting by comparing train vs test performance
- ✅ Checking for bias across demographic groups
- ✅ Generating a full classification report

**Further reading:**
- [scikit-learn metrics guide](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [Google ML Fairness guide](https://developers.google.com/machine-learning/crash-course/fairness/overview)
- [Amazon hiring bias case study](https://www.reuters.com/article/us-amazon-com-jobs-automation-insight/amazon-scraps-secret-ai-recruiting-tool-that-showed-bias-against-women-idUSKCN1MK08G)